# Detections Classes Replacement - No Training Demo

This notebook satisfies the `Model Train Notebook (Colab/Local)` checklist item for the NovaVision Detections Classes Replacement component.

No model training is required for this package. The component does not train a detector or classifier and does not run neural network inference. It is deterministic post-processing logic: it receives detection predictions and classification predictions from upstream workflow blocks, then replaces each detection's generic class fields with the matching classifier result.

## Package Input / Output Idea

The NovaVision component model uses these high-level fields:

- `inputImage`: image data passed through from the upstream image flow.
- `inputDetections`: detection dictionaries with `detection_id`, bounding box fields, `confidence`, `class`, `class_id`, and `metadata`.
- `inputData`: classification dictionaries, strings, or string lists.
- `outputImages`: unchanged image data copied from `inputImage`.
- `outputDetections`: updated detection dictionaries where the bounding box is preserved and class fields are replaced.

The example below mirrors the intended package behavior with small synthetic data.

In [ ]:
from copy import deepcopy
import json
import uuid

object_detection_predictions = [
    {
        "detection_id": "det-vehicle-1",
        "x": 120.0,
        "y": 80.0,
        "width": 64.0,
        "height": 40.0,
        "confidence": 0.82,
        "class": "vehicle",
        "class_id": 1,
        "metadata": {"source": "generic-detector"},
    },
    {
        "detection_id": "det-animal-1",
        "x": 260.0,
        "y": 140.0,
        "width": 72.0,
        "height": 58.0,
        "confidence": 0.76,
        "class": "animal",
        "class_id": 2,
        "metadata": {"source": "generic-detector"},
    },
]

classification_predictions = [
    {
        "parent_id": "det-vehicle-1",
        "top": "truck",
        "class_id": 7,
        "confidence": 0.94,
    },
    {
        "parent_id": "det-animal-1",
        "predictions": [
            {"class": "cat", "class_id": 12, "confidence": 0.21},
            {"class": "golden_retriever", "class_id": 18, "confidence": 0.89},
        ],
    },
]

In [ ]:
def extract_replacement_label(classification):
    """Return class name, class ID, and confidence from one classifier output."""
    if isinstance(classification, str):
        return classification, 0, 1.0

    if isinstance(classification, list):
        label = next((item for item in classification if isinstance(item, str) and item), "")
        return label, 0, 1.0

    if "top" in classification:
        return (
            classification["top"],
            classification.get("class_id", 0),
            classification.get("confidence", 1.0),
        )

    predictions = classification.get("predictions", [])
    best = max(predictions, key=lambda item: item.get("confidence", 0.0))
    return (
        best.get("class", best.get("class_name", "")),
        best.get("class_id", 0),
        best.get("confidence", 1.0),
    )


def replace_detection_classes(detections, classifications):
    classifications_by_parent_id = {
        item["parent_id"]: item
        for item in classifications
        if isinstance(item, dict) and item.get("parent_id")
    }
    updated_detections = []

    for detection in detections:
        original_detection_id = detection["detection_id"]
        classification = classifications_by_parent_id.get(original_detection_id)
        if classification is None:
            continue

        class_name, class_id, confidence = extract_replacement_label(classification)
        updated = deepcopy(detection)
        updated["detection_id"] = str(uuid.uuid4())
        updated["class"] = class_name
        updated["class_id"] = class_id
        updated["confidence"] = confidence
        updated.setdefault("metadata", {})["original_detection_id"] = original_detection_id
        updated_detections.append(updated)

    return updated_detections

In [ ]:
predictions = replace_detection_classes(
    object_detection_predictions,
    classification_predictions,
)

print("Input detections:")
print(json.dumps(object_detection_predictions, indent=2))
print("\nClassification predictions:")
print(json.dumps(classification_predictions, indent=2))
print("\nUpdated detections:")
print(json.dumps(predictions, indent=2))

assert predictions[0]["class"] == "truck"
assert predictions[0]["class_id"] == 7
assert predictions[0]["confidence"] == 0.94
assert predictions[0]["x"] == object_detection_predictions[0]["x"]
assert predictions[0]["metadata"]["original_detection_id"] == "det-vehicle-1"

assert predictions[1]["class"] == "golden_retriever"
assert predictions[1]["class_id"] == 18
assert predictions[1]["confidence"] == 0.89
assert predictions[1]["width"] == object_detection_predictions[1]["width"]
assert predictions[1]["metadata"]["original_detection_id"] == "det-animal-1"

## Result

The generic detection classes `vehicle` and `animal` are replaced by classifier outputs `truck` and `golden_retriever`. The bounding box fields stay unchanged, and each updated detection receives metadata pointing back to the original detection ID.

This validates the learning/demo requirement without pretending to train a model that the component does not need.